In [87]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [88]:
class Value:
    def __init__(self, data, _children=(), _op='', _label=''):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._op = _op
        self._label = _label

    def __repr__(self):
        return f"Value(data = {self.data})"

    def __add__(self, other):
        return Value(self.data + other.data, (self, other), '+')

    def __mul__(self, other):
        return Value(self.data * other.data, (self, other), '*')

In [89]:
a = Value(3, _label='a')
b = Value(-2, _label='b')
c = Value(5, _label='c')

In [90]:
a * b

Value(data = -6)

In [91]:
a + b

Value(data = 1)

In [92]:
d = a * b; d._label = 'd'
e = d + c; e._label = 'e'
f = Value(-1, _label='f')
L = e * f; L._label = 'L'
L

Value(data = 1)

In [93]:
e._prev

{Value(data = -6), Value(data = 5)}

In [94]:
e._op

'+'

In [95]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f } grad %.4f }" % (n._label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [96]:
draw_dot(L)

Error: bad label format { d | data -6.0000 } grad 0.0000 }
Error: bad label format { L | data 1.0000 } grad 0.0000 }
Error: bad label format { f | data -1.0000 } grad 0.0000 }
Error: bad label format { a | data 3.0000 } grad 0.0000 }
Error: bad label format { b | data -2.0000 } grad 0.0000 }
Error: bad label format { e | data -1.0000 } grad 0.0000 }
Error: bad label format { c | data 5.0000 } grad 0.0000 }


CalledProcessError: Command '[PosixPath('dot'), '-Kdot', '-Tsvg']' returned non-zero exit status 1. [stderr: 'Error: bad label format { d | data -6.0000 } grad 0.0000 }\nError: bad label format { L | data 1.0000 } grad 0.0000 }\nError: bad label format { f | data -1.0000 } grad 0.0000 }\nError: bad label format { a | data 3.0000 } grad 0.0000 }\nError: bad label format { b | data -2.0000 } grad 0.0000 }\nError: bad label format { e | data -1.0000 } grad 0.0000 }\nError: bad label format { c | data 5.0000 } grad 0.0000 }\n']